***
# Merge BTS, ASPM, and NOAA data

Create one departure-level table for the selected airport and year. BTS supplies the flight records and scheduled departure timestamp. ASPM supplies conditions from the previous completed airport hour. NOAA supplies the most recent weather report available at or before scheduled departure.

The joins keep the matched source timestamps so their timing can be audited and future information can be detected.
***

In [1]:
YEAR = 2019

AIRPORT = "JFK"
# AIRPORT = "ORD"
# AIRPORT = "ATL"

BTS_DATA_FILE = f"../data/bts/cleaned/{AIRPORT}_{YEAR}.csv"

ASPM_DATA_FILE = f"../data/aspm/cleaned/{AIRPORT}_{YEAR}.csv"

NOAA_DATA_FILE = f"../data/noaa/cleaned/{AIRPORT}_{YEAR}.csv"

MERGED_DATA_FILE = f"../data/merged/{AIRPORT}_{YEAR}.csv"

NOAA_TOLERANCE = "90min"

***
## Load the cleaned source files

Load the validated BTS, ASPM, and NOAA files. Parse the timestamp columns during loading so the joins use real datetimes rather than text.
***

In [2]:
from pathlib import Path

import pandas as pd

NOAA_TOLERANCE = pd.Timedelta(NOAA_TOLERANCE)

required_files = [BTS_DATA_FILE, ASPM_DATA_FILE, NOAA_DATA_FILE]
missing_files = [path for path in required_files if not Path(path).is_file()]

if missing_files:
    raise FileNotFoundError(
        "Run the clean notebooks first. Missing files: " + ", ".join(missing_files)
    )

bts = pd.read_csv(BTS_DATA_FILE, parse_dates=["FlightDate", "DATE"])
aspm = pd.read_csv(ASPM_DATA_FILE, parse_dates=["report_date", "DATE"])
noaa = pd.read_csv(NOAA_DATA_FILE, parse_dates=["DATE"])

pd.DataFrame({
    "rows": [len(bts), len(aspm), len(noaa)],
    "columns": [len(bts.columns), len(aspm.columns), len(noaa.columns)]
}, index=["BTS", "ASPM", "NOAA"])

,rows,columns
BTS,249428,28
ASPM,8760,10
NOAA,13094,18


***
## Validate the merge keys

Confirm that the required airport and timestamp fields are present and contain valid dates. ASPM must have one row per airport-hour. Exact duplicate NOAA timestamps are reported but retained; a stable sort means the last same-time report in source order is used by the backward as-of join.
***

In [3]:
required_columns = {
    "BTS": {"DATE", "FlightDate", "Origin", "Dest"},
    "ASPM": {"DATE", "airport", "report_date", "Hour"},
    "NOAA": {"DATE"}
}
source_frames = {"BTS": bts, "ASPM": aspm, "NOAA": noaa}

missing_columns = {
    name: sorted(columns - set(source_frames[name].columns))
    for name, columns in required_columns.items()
    if columns - set(source_frames[name].columns)
}

if missing_columns:
    raise KeyError(f"Missing merge columns: {missing_columns}")

invalid_dates = pd.Series({
    "BTS missing DATE": int(bts["DATE"].isna().sum()),
    "ASPM missing DATE": int(aspm["DATE"].isna().sum()),
    "NOAA missing DATE": int(noaa["DATE"].isna().sum())
})

if invalid_dates.sum() > 0:
    raise ValueError(f"Invalid merge timestamps:\n{invalid_dates}")

aspm_duplicate_keys = int(
    aspm.duplicated(subset=["airport", "DATE"], keep=False).sum()
)
if aspm_duplicate_keys > 0:
    raise ValueError(f"ASPM contains {aspm_duplicate_keys} duplicate airport-hour rows")

noaa_duplicate_timestamps = int(noaa["DATE"].duplicated(keep=False).sum())

pd.Series({
    "ASPM duplicate airport-hour rows": aspm_duplicate_keys,
    "NOAA rows with duplicate timestamps": noaa_duplicate_timestamps
}, name="merge-key validation")

ASPM duplicate airport-hour rows       0
NOAA rows with duplicate timestamps    8
Name: merge-key validation, dtype: int64

***
## Prepare airport departures and source timestamps

Keep flights departing from the selected airport because the available ASPM and NOAA records describe conditions at that origin. Rename the ASPM and NOAA timestamps so the matched source times remain clear in the merged table.
***

In [4]:
bts_departures = bts.loc[bts["Origin"] == AIRPORT].copy()
bts_departures = bts_departures.sort_values("DATE", kind="stable").reset_index(drop=True)

aspm = aspm.rename(columns={
    "airport": "ASPM_Airport",
    "report_date": "ASPM_ReportDate",
    "Hour": "ASPM_Hour",
    "DATE": "ASPM_DATE"
})
aspm_feature_columns = [
    column for column in aspm.columns
    if column not in {"ASPM_Airport", "ASPM_ReportDate", "ASPM_Hour", "ASPM_DATE"}
]
aspm = aspm.rename(columns={
    column: f"ASPM_{column.replace(' ', '_')}"
    for column in aspm_feature_columns
})

noaa = noaa.rename(columns={"DATE": "NOAA_DATE"})
noaa = noaa.sort_values("NOAA_DATE", kind="stable").reset_index(drop=True)

pd.Series({
    "all BTS rows": len(bts),
    f"{AIRPORT} departure rows": len(bts_departures),
    "non-origin rows excluded": len(bts) - len(bts_departures)
}, name="departure selection")

all BTS rows                249428
JFK departure rows          124759
non-origin rows excluded    124669
Name: departure selection, dtype: int64

***
## Join the previous ASPM hour

For each scheduled departure, use the ASPM record from the previous clock hour. For example, a 14:35 departure uses the 13:00 ASPM record. This prevents current-hour realized airport delays from leaking into the prediction inputs.
***

In [5]:
bts_departures["ASPM_LOOKUP_DATE"] = (
    bts_departures["DATE"].dt.floor("h") - pd.Timedelta(hours=1)
)

merged = bts_departures.merge(
    aspm,
    left_on=["Origin", "ASPM_LOOKUP_DATE"],
    right_on=["ASPM_Airport", "ASPM_DATE"],
    how="left",
    validate="many_to_one"
)

merged["ASPM_AGE_MINUTES"] = (
    merged["DATE"] - merged["ASPM_DATE"]
).dt.total_seconds().div(60)

merged[["DATE", "ASPM_LOOKUP_DATE", "ASPM_DATE", "ASPM_AGE_MINUTES"]].head()

,DATE,ASPM_LOOKUP_DATE,ASPM_DATE,ASPM_AGE_MINUTES
0,2019-01-01 05:00:00,2019-01-01 04:00:00,2019-01-01 04:00:00,60.0
1,2019-01-01 05:00:00,2019-01-01 04:00:00,2019-01-01 04:00:00,60.0
2,2019-01-01 05:00:00,2019-01-01 04:00:00,2019-01-01 04:00:00,60.0
3,2019-01-01 05:40:00,2019-01-01 04:00:00,2019-01-01 04:00:00,100.0
4,2019-01-01 05:45:00,2019-01-01 04:00:00,2019-01-01 04:00:00,105.0


***
## Join the latest available NOAA report

Use a backward as-of join to select the most recent NOAA observation at or before scheduled departure. The tolerance prevents a stale weather report from being carried forward more than 90 minutes. No future weather observation is allowed.
***

In [6]:
merged = merged.sort_values("DATE", kind="stable").reset_index(drop=True)

merged = pd.merge_asof(
    merged,
    noaa,
    left_on="DATE",
    right_on="NOAA_DATE",
    direction="backward",
    tolerance=NOAA_TOLERANCE,
    allow_exact_matches=True
)

merged["NOAA_AGE_MINUTES"] = (
    merged["DATE"] - merged["NOAA_DATE"]
).dt.total_seconds().div(60)

merged[["DATE", "NOAA_DATE", "NOAA_AGE_MINUTES"]].head()

,DATE,NOAA_DATE,NOAA_AGE_MINUTES
0,2019-01-01 05:00:00,2019-01-01 04:51:00,9.0
1,2019-01-01 05:00:00,2019-01-01 04:51:00,9.0
2,2019-01-01 05:00:00,2019-01-01 04:51:00,9.0
3,2019-01-01 05:40:00,2019-01-01 04:51:00,49.0
4,2019-01-01 05:45:00,2019-01-01 04:51:00,54.0


***
## Validate the merged data

Confirm that both joins preserve the number of departure rows, ASPM matches the requested previous hour, NOAA never comes from the future, and source ages stay within the intended limits. Missing source matches are reported rather than silently removed.
***

In [7]:
merge_validation = pd.Series({
    "BTS departure rows": len(bts_departures),
    "merged rows": len(merged),
    "row-count difference": len(merged) - len(bts_departures),
    "missing ASPM matches": int(merged["ASPM_DATE"].isna().sum()),
    "ASPM timestamp mismatches": int((
        merged["ASPM_DATE"].notna()
        & (merged["ASPM_DATE"] != merged["ASPM_LOOKUP_DATE"])
    ).sum()),
    "missing NOAA matches": int(merged["NOAA_DATE"].isna().sum()),
    "future NOAA matches": int((merged["NOAA_DATE"] > merged["DATE"]).sum()),
    "NOAA matches older than tolerance": int((
        merged["NOAA_AGE_MINUTES"] > NOAA_TOLERANCE.total_seconds() / 60
    ).sum()),
    "rows with duplicated BTS flight keys": int(merged.duplicated(
        subset=[
            "FlightDate", "Reporting_Airline",
            "Flight_Number_Reporting_Airline", "Origin", "Dest", "CRSDepTime"
        ],
        keep=False
    ).sum())
}, name="merged-data validation")

merge_validation

BTS departure rows                      124759
merged rows                             124759
row-count difference                         0
missing ASPM matches                         0
ASPM timestamp mismatches                    0
missing NOAA matches                         0
future NOAA matches                          0
NOAA matches older than tolerance            0
rows with duplicated BTS flight keys         0
Name: merged-data validation, dtype: int64

In [8]:
merged[["ASPM_AGE_MINUTES", "NOAA_AGE_MINUTES"]].describe().T

,count,mean,std,min,25%,50%,75%,max
ASPM_AGE_MINUTES,124759.0,88.896753,18.877087,60.0,73.0,90.0,105.0,119.0
NOAA_AGE_MINUTES,124759.0,24.426061,16.846088,0.0,9.0,24.0,39.0,59.0


***
## Save the merged data

Create the merged-data directory when needed and save the departure-level dataframe without the pandas index. Source timestamps and age fields are retained for later auditing and feature selection.
***

In [9]:
Path(MERGED_DATA_FILE).parent.mkdir(parents=True, exist_ok=True)
merged.to_csv(MERGED_DATA_FILE, index=False)

In [11]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 124759 entries, 0 to 124758
Data columns (total 59 columns):
 #   Column                           Non-Null Count   Dtype         
---  ------                           --------------   -----         
 0   Year                             124759 non-null  int64         
 1   Quarter                          124759 non-null  int64         
 2   Month                            124759 non-null  int64         
 3   DayofMonth                       124759 non-null  int64         
 4   DayOfWeek                        124759 non-null  int64         
 5   FlightDate                       124759 non-null  datetime64[ns]
 6   Reporting_Airline                124759 non-null  object        
 7   Tail_Number                      124759 non-null  object        
 8   Flight_Number_Reporting_Airline  124759 non-null  int64         
 9   Origin                           124759 non-null  object        
 10  OriginState                      124759 non-